In [50]:
pip install pyro-ppl

In [51]:
import math
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import pyro
import pyro.distributions as dist
from pyro.nn import PyroModule, PyroSample
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.optim import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os

In [52]:
TRAIN_CSV = 'stat_walls_full_Re100.csv'

In [53]:
df = pd.read_csv(TRAIN_CSV)
clean_data = []
for i in range(len(df)):
  cell = df.iloc[i, 0]
  parts = cell.split()
  floats = [float(x) for x in parts]
  clean_data.append(floats)
df= pd.DataFrame(clean_data)
df.columns= ['cel_number', 'x-coordinate', 'y-coordinate', 'y-wall-shear', ' x-wall-shear', 'wall-shear', 'velocity-magnitude', 'y-velocity', ' x-velocity', 'pressure', 'y-coordinate', 'x-coordinate']


In [54]:
print(df.columns)

Index(['cel_number', 'x-coordinate', 'y-coordinate', 'y-wall-shear',
       ' x-wall-shear', 'wall-shear', 'velocity-magnitude', 'y-velocity',
       ' x-velocity', 'pressure', 'y-coordinate', 'x-coordinate'],
      dtype='object')


In [55]:
print(df)

     cel_number  x-coordinate  y-coordinate  y-wall-shear   x-wall-shear  \
0           1.0      0.352406      0.996124  2.861188e-07           -0.0   
1           2.0      0.352406      0.988372  1.305257e-07           -0.0   
2           3.0      0.352406      0.980620  1.062239e-07           -0.0   
3           4.0      0.352406      0.972868  8.294051e-08           -0.0   
4           5.0      0.352406      0.965116  6.769041e-08           -0.0   
..          ...           ...           ...           ...            ...   
382       383.0      1.352406      0.965116 -6.945936e-08           -0.0   
383       384.0      1.352406      0.972868 -8.465775e-08           -0.0   
384       385.0      1.352406      0.980620 -1.079131e-07           -0.0   
385       386.0      1.352406      0.988372 -1.320214e-07           -0.0   
386       387.0      1.352406      0.996124 -2.860065e-07           -0.0   

       wall-shear  velocity-magnitude  y-velocity   x-velocity      pressure  \
0    2.

In [56]:
FEATURE_COLS = [c for c in df.columns if c != TARGET_COL and c.lower() != "cellnumber"]
TARGET_COL = "wall-shear"

In [74]:
print(df['wall-shear'])

0      2.861188e-07
1      1.305257e-07
2      1.062239e-07
3      8.294051e-08
4      6.769041e-08
           ...     
382    6.945936e-08
383    8.465775e-08
384    1.079131e-07
385    1.320214e-07
386    2.860065e-07
Name: wall-shear, Length: 387, dtype: float64


In [57]:
X = df[FEATURE_COLS].values.astype("float32")
y = df[TARGET_COL].values.astype("float32").reshape(-1, 1)

In [58]:
X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED)

In [59]:
x_scaler = StandardScaler().fit(X_train)
y_scaler = StandardScaler().fit(y_train)

X_train_s = x_scaler.transform(X_train).astype("float32")
y_train_s = y_scaler.transform(y_train).astype("float32")

X_test_s = x_scaler.transform(X_test).astype("float32")
y_test_s = y_scaler.transform(y_test).astype("float32")

In [60]:
BATCH_SIZE = 512
LR = 1e-3
EPOCHS = 2000
HIDDEN = 128
SEED = 0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
pyro.set_rng_seed(SEED)

In [61]:
train_ds = TensorDataset(torch.from_numpy(X_train_s), torch.from_numpy(y_train_s))
test_ds = TensorDataset(torch.from_numpy(X_test_s), torch.from_numpy(y_test_s))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

in_dim = X_train_s.shape[1]
out_dim = 1

In [62]:
class BayesianRegression(PyroModule):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.fc1 = PyroModule[nn.Linear](in_dim, hidden_dim)
        self.fc1.weight = PyroSample(lambda prior: dist.Normal(
            torch.zeros_like(prior), torch.ones_like(prior)
        ).independent(1))
        self.fc1.bias = PyroSample(lambda prior: dist.Normal(
            torch.zeros_like(prior), torch.ones_like(prior)
        ).independent(1))

        self.fc2 = PyroModule[nn.Linear](hidden_dim, out_dim)
        self.fc2.weight = PyroSample(lambda prior: dist.Normal(
            torch.zeros_like(prior), torch.ones_like(prior)
        ).independent(1))
        self.fc2.bias = PyroSample(lambda prior: dist.Normal(
            torch.zeros_like(prior), torch.ones_like(prior)
        ).independent(1))

        self.sigma = PyroSample(dist.Uniform(1e-6, 10.0))


In [63]:
    def forward(self, x, y=None):
        x = torch.relu(self.fc1(x))
        mu = self.fc2(x).squeeze(-1)
        sigma = torch.abs(self.sigma)

        with pyro.plate("data", x.shape[0]):
            return pyro.sample(
                "obs",
                dist.Normal(mu, sigma),
                obs=None if y is None else y.squeeze(-1),
            )

In [72]:

# --- STEP 1: DEFINE THE FIXED MODEL ---
class BayesianRegression(PyroModule):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()

        # Layer 1
        self.fc1 = PyroModule[nn.Linear](in_dim, hidden_dim)
        # Weight shape: [hidden_dim, in_dim]
        self.fc1.weight = PyroSample(
            dist.Normal(0., 1.).expand([hidden_dim, in_dim]).to_event(2)
        )
        # Bias shape: [hidden_dim]
        self.fc1.bias = PyroSample(
            dist.Normal(0., 1.).expand([hidden_dim]).to_event(1)
        )

        # Layer 2
        self.fc2 = PyroModule[nn.Linear](hidden_dim, out_dim)
        # Weight shape: [out_dim, hidden_dim]
        self.fc2.weight = PyroSample(
            dist.Normal(0., 1.).expand([out_dim, hidden_dim]).to_event(2)
        )
        # Bias shape: [out_dim]
        self.fc2.bias = PyroSample(
            dist.Normal(0., 1.).expand([out_dim]).to_event(1)
        )

        # Output Noise
        self.sigma = PyroSample(dist.Uniform(1e-5, 10.0))

    def forward(self, x, y=None):
        x = self.fc1(x)
        x = torch.relu(x)
        mu = self.fc2(x).squeeze()

        sigma = self.sigma

        with pyro.plate("data", x.shape[0]):
            pyro.sample("obs", dist.Normal(mu, sigma), obs=y.squeeze() if y is not None else None)
        return mu

# --- STEP 2: CLEAR OLD MEMORY & SETUP ---
pyro.clear_param_store() # <--- Crucial! Wipes the old broken params
bayes_nn = BayesianRegression(in_dim, 64, out_dim).to(DEVICE)
guide = AutoDiagonalNormal(bayes_nn)
optimizer = pyro.optim.Adam({"lr": 0.01})
svi = SVI(bayes_nn, guide, optimizer, loss=Trace_ELBO())

# --- STEP 3: TRAIN ---
print("Training New Model...")
for epoch in range(1000):
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        total_loss += svi.step(xb, yb)
    if (epoch + 1) % 200 == 0:
        print(f"Epoch {epoch+1} | Loss: {total_loss / len(train_loader.dataset):.4f}")

# --- STEP 4: PREDICT & EVALUATE ---
print("\nPredicting...")
bayes_nn.eval()
predictive = Predictive(bayes_nn, guide=guide, num_samples=200, return_sites=("obs",))

X_test_t = torch.from_numpy(X_test_s).to(DEVICE)
# Get samples and DETACH from graph
samples = predictive(X_test_t)["obs"].detach().cpu().numpy()

# Calculate stats in SCALED space
pred_mean_scaled = samples.mean(axis=0)
pred_std_scaled = samples.std(axis=0)

# Inverse transform to REAL space
pred_mean = y_scaler.inverse_transform(pred_mean_scaled.reshape(-1, 1)).flatten()
pred_std = pred_std_scaled * y_scaler.scale_[0]

y_true = y_test.flatten()
rmse = ((pred_mean - y_true) ** 2).mean() ** 0.5
print(f"Test RMSE: {rmse:.4f}")

print("\nSample Results:")
for i in range(5):
    print(f"Pred: {pred_mean[i]:.3f} ± {pred_std[i]:.3f} | True: {y_true[i]:.3f}")

Training New Model...
Epoch 200 | Loss: 4.1934
Epoch 400 | Loss: 2.9196
Epoch 600 | Loss: 2.2380
Epoch 800 | Loss: 1.9649
Epoch 1000 | Loss: 2.3158

Predicting...
Test RMSE: 0.0000

Sample Results:
Pred: 0.000 ± 0.000 | True: 0.000
Pred: 0.000 ± 0.000 | True: 0.000
Pred: -0.000 ± 0.000 | True: 0.000
Pred: 0.000 ± 0.000 | True: 0.000
Pred: 0.000 ± 0.000 | True: 0.000


In [77]:
# 1. Find the row for Cell 382
# NOTE: Check your column name! Based on your image, it looks like 'cel_number'
# We use .iloc[0] to grab the single row as a Series, then convert back to 2D
target_cell_id = 382
row_data = df[df['cel_number'] == target_cell_id]

if row_data.empty:
    print(f"Error: Cell {target_cell_id} not found in DataFrame.")
else:
    # 2. Extract ONLY the features used for training
    # We re-use the FEATURE_COLS list from the start of your code
    x_382_raw = row_data[FEATURE_COLS].values.astype("float32")

    # 3. Scale the features (CRITICAL STEP)
    # The model only understands "scaled" values (roughly -1 to 1)
    x_382_scaled = x_scaler.transform(x_382_raw)

    # 4. Convert to Tensor and move to GPU/CPU
    x_382_tensor = torch.from_numpy(x_382_scaled).to(DEVICE)

    # 5. Run Prediction
    bayes_nn.eval()
    # We take 500 samples to get a smooth probability distribution
    predictive = Predictive(bayes_nn, guide=guide, num_samples=500, return_sites=("obs",))
    samples_382 = predictive(x_382_tensor)["obs"].detach().cpu().numpy()

    # 6. Unscale the results back to real units
    # Calculate Mean in scaled space
    mean_scaled = samples_382.mean()
    # Inverse transform requires a 2D shape like [[value]]
    pred_val = y_scaler.inverse_transform([[mean_scaled]])[0][0]

    # Calculate Std Dev (Uncertainty)
    # We multiply by the scaler's scale factor to return to real units
    std_scaled = samples_382.std()
    pred_std = std_scaled * y_scaler.scale_[0]

    # 7. Print using Scientific Notation (:.3e)
    true_val = row_data[TARGET_COL].values[0]

    print(f"--- Results for Cell {target_cell_id} ---")
    print(f"Predicted: {pred_val:.4e} ± {pred_std:.4e}")
    print(f"Actual:    {true_val:.4e}")

    # Calculate Error
    error = abs(pred_val - true_val)
    print(f"Diff:      {error:.4e}")

--- Results for Cell 382 ---
Predicted: 4.1459e-08 ± 3.5626e-08
Actual:    5.8031e-08
Diff:      1.6572e-08
